# Feature Engineering
## Cryptocurrency Liquidity Prediction

Creating liquidity-related features for model training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('Libraries imported!')

Libraries imported!


## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/crypto_processed.csv')
print(f'Dataset: {df.shape}')
df.head()

Dataset: (1000, 12)


,coin,symbol,price,change_1h,change_24h,change_7d,volume_24h,market_cap,date,price_normalized,volume_24h_normalized,market_cap_normalized
0,Bitcoin,BTC,40859.460000,0.022,0.030,0.055,3.539076e+10,7.709915e+11,2022-03-16,0.991319,0.610870,0.993446
1,Ethereum,ETH,2744.410000,0.024,0.034,0.065,1.974870e+10,3.271044e+11,2022-03-16,0.066584,0.340877,0.421435
2,Tether,USDT,1.000000,-0.001,-0.001,0.000,5.793497e+10,7.996516e+10,2022-03-16,0.000024,1.000000,0.102962
3,BNB,BNB,383.430000,0.018,0.028,0.004,1.395854e+09,6.404382e+10,2022-03-16,0.009303,0.024093,0.082445
4,USD Coin,USDC,0.999874,-0.001,0.000,-0.000,3.872274e+09,5.222214e+10,2022-03-16,0.000024,0.066838,0.067211


## 2. Liquidity Ratio (Target)

In [11]:
# Primary target: Volume / Market Cap
df['liquidity_ratio'] = df['volume_24h'] / df['market_cap']
df['liquidity_ratio'] = df['liquidity_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0)

print('Liquidity Ratio Stats:')
print(df['liquidity_ratio'].describe())

Liquidity Ratio Stats:
count    1000.000000
mean        0.103132
std         0.365718
min         0.000000
25%         0.008709
50%         0.033588
75%         0.086624
max         5.948545
Name: liquidity_ratio, dtype: float64


## 3. Volatility Score

In [12]:
change_cols = ['change_1h', 'change_24h', 'change_7d']
available = [c for c in change_cols if c in df.columns]

df['volatility_score'] = df[available].std(axis=1)
df['avg_abs_change'] = df[available].abs().mean(axis=1)

print(f"Volatility Mean: {df['volatility_score'].mean():.4f}")

Volatility Mean: 0.0510


## 4. Turnover Rate

In [13]:
max_vol = df['volume_24h'].max()
df['turnover_rate'] = df['volume_24h'] / max_vol

print(f"Turnover Rate Range: {df['turnover_rate'].min():.4f} - {df['turnover_rate'].max():.4f}")

Turnover Rate Range: 0.0000 - 1.0000


## 5. Market Dominance

In [14]:
total_mcap = df['market_cap'].sum()
df['market_dominance'] = (df['market_cap'] / total_mcap) * 100

print(f"Max Dominance: {df['market_dominance'].max():.2f}%")
print(f"Top 10 Total: {df.nlargest(10, 'market_dominance')['market_dominance'].sum():.2f}%")

Max Dominance: 20.67%
Top 10 Total: 69.44%


## 6. Momentum Indicators

In [15]:
if 'change_1h' in df.columns:
    df['momentum_short'] = df['change_1h']
if 'change_24h' in df.columns:
    df['momentum_medium'] = df['change_24h']
if 'change_7d' in df.columns:
    df['momentum_long'] = df['change_7d']

print('Momentum features added!')

Momentum features added!


## 7. Liquidity Classification

In [16]:
q25 = df['liquidity_ratio'].quantile(0.25)
q75 = df['liquidity_ratio'].quantile(0.75)

conditions = [
    df['liquidity_ratio'] >= q75,
    df['liquidity_ratio'] >= q25,
    df['liquidity_ratio'] < q25
]
choices = ['High', 'Medium', 'Low']
df['liquidity_class'] = np.select(conditions, choices, default='Medium')

print('Classification:')
print(df['liquidity_class'].value_counts())

Classification:
liquidity_class
Medium    500
High      250
Low       250
Name: count, dtype: int64


## 8. Feature Summary

In [17]:
new_features = ['liquidity_ratio', 'volatility_score', 'avg_abs_change', 
                'turnover_rate', 'market_dominance', 'liquidity_class']

print('=== NEW FEATURES ===')
for f in new_features:
    if f in df.columns:
        if df[f].dtype in ['float64', 'int64']:
            print(f"{f}: Mean={df[f].mean():.4f}, Std={df[f].std():.4f}")
        else:
            print(f"{f}: {df[f].value_counts().to_dict()}")

=== NEW FEATURES ===
liquidity_ratio: Mean=0.1031, Std=0.3657
volatility_score: Mean=0.0510, Std=0.1170
avg_abs_change: Mean=0.0432, Std=0.0847
turnover_rate: Mean=0.0050, Std=0.0476
market_dominance: Mean=0.1000, Std=1.0129
liquidity_class: {'Medium': 500, 'High': 250, 'Low': 250}


In [18]:
# Correlation with target
feature_cols = ['price', 'volume_24h', 'market_cap', 'volatility_score', 'turnover_rate', 'market_dominance']
available = [c for c in feature_cols if c in df.columns]

corr = df[available + ['liquidity_ratio']].corr()['liquidity_ratio'].drop('liquidity_ratio').sort_values(ascending=False)
print('\nCorrelation with Liquidity Ratio:')
print(corr)


Correlation with Liquidity Ratio:
volatility_score    0.427294
turnover_rate       0.079741
volume_24h          0.079741
market_cap         -0.007195
market_dominance   -0.007195
price              -0.029187
Name: liquidity_ratio, dtype: float64


## 9. Save Featured Data

In [19]:
df.to_csv('../data/processed/crypto_featured.csv', index=False)
print(f'Saved! Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Saved! Shape: (1000, 21)
Columns: ['coin', 'symbol', 'price', 'change_1h', 'change_24h', 'change_7d', 'volume_24h', 'market_cap', 'date', 'price_normalized', 'volume_24h_normalized', 'market_cap_normalized', 'liquidity_ratio', 'volatility_score', 'avg_abs_change', 'turnover_rate', 'market_dominance', 'momentum_short', 'momentum_medium', 'momentum_long', 'liquidity_class']
